# Problem 7: The Gene Finder Grand Prix

---

> Place all files for this problem in a directory called `problem_7`.

## Overview

In Module 2, you built a gene finder that combines ORF detection and CpG island analysis to identify candidate genes on Human Chromosome 21. It works — but it was written to be correct, not fast. Chromosome 21 is approximately **46.7 million base pairs**, and your serial implementation likely takes a while to get through it.

In this module you have learned how to run jobs on a cluster, use `srun` for multi-node execution, submit array jobs, chain jobs with dependencies, and parallelize with Python `multiprocessing`. Now it's time to put it all together...

**Make your gene finder as fast as possible.**

There are no restrictions on your approach. You may use any combination of techniques from this module — or invent your own. The only rule is that your optimized program must produce the **same results** as your original serial implementation.

**There will be a prize for the fastest wall-clock time to process Chromosome 21.**


## Getting Chromosome 21

Use the full chromosome, NCBI's `NC_000021.9` (GRCh38): 46,709,983 bases, a 47 MB file. If you still have your Module 2 download, use that. Otherwise download it on a Midway3 login node (compute nodes have no internet access):

```bash
mkdir -p data/chromosome-21
curl -sSf -o data/chromosome-21/chr21.fasta \
  "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000021.9&rettype=fasta&retmode=text"
grep -v '>' data/chromosome-21/chr21.fasta | tr -d '\n' | wc -c    # 46709983
```

The file is too big for git, so `.gitignore` already excludes it: don't commit it. NCBI's page for the record is <https://www.ncbi.nlm.nih.gov/nuccore/NC_000021.9>.

## What Your Gene Finder Computes

Your optimized version must follow the same rules as your Module 2 gene finder, so it gives the same answer.

**Open reading frames** (Module 1, Problem 4)

- Search all **six reading frames**: three on the forward strand and three on the reverse complement
- An ORF begins at an `ATG` start codon and ends at the first stop codon (`TAA`, `TAG` or `TGA`) that follows it **in the same frame**
- Every `ATG` begins an ORF, including one inside another ORF; a start codon with no stop codon after it is not an ORF

**CpG islands** (Module 2, Problem 3, the Gardiner-Garden and Frommer method)

- Slide a **200 bp window** along the sequence, **1 bp at a time**; the window at position *i* covers bases *i* to *i* + 199
- For each window:
  - **GC-content** = (count of G + count of C) / 200 × 100
  - **Observed** = the number of `CG` dinucleotides (a `C` immediately followed by a `G`)
  - **Expected** = (count of C × count of G) / 200
  - **Obs/Exp** = Observed / Expected
- A window is part of a CpG island if **both** Obs/Exp > 0.6 **and** GC-content > 50%
- `N` (an unknown base) is not counted as any base, but the window still spans 200 positions

**Candidate genes** (Module 2, Problem 4): an ORF that

1. starts with `ATG`,
2. is at least 96 bp long (32 amino acids), counting the start and stop codons,
3. overlaps at least one CpG island, by any amount, and
4. has that island overlap its **first third**: from *start* to *start* + (*end* − *start*) / 3

Watch the windows when you split the sequence: a 200 bp window, an ORF or an island can cross the boundary between two pieces.

## The Rules

1. **Correctness first.** Your optimized gene finder must produce the same candidate genes as your original Module 2 implementation. If your results differ, your submission will not be eligible for the competition.

2. **Wall-clock time is king.** The metric is total wall time from job submission to final output — including any time spent in the queue, staging data, combining results, or running dependency jobs. Use `sacct` to measure this.

3. **Document everything.** Your `README.md` must explain your approach clearly enough that someone else could reproduce your results.

4. **Fair play.** All submissions run under the same `mpcs56430` account on the `caslake` partition. Do not hardcode node names or attempt to game the scheduler.

5. **Resource limit.** Your solution may use a maximum of **10 nodes** and **48 cores per node** (480 cores total). Array jobs are limited to a maximum of **50 tasks**. This keeps the competition fair and ensures we don't consume the entire class allocation in one run.


## Approaches to Consider

There is no single "correct" strategy — the best approach will depend on how you decompose the problem. Here are some directions to explore. You may combine multiple techniques.

### Decompose by Reading Frame
Your gene finder searches 6 reading frames (3 forward, 3 reverse complement). Each frame is independent. You could assign each frame to a separate process, array task, or node.

### Decompose by Sequence Region
Chromosome 21 is 46.7 Mbp. You could split the sequence into chunks and process each chunk in parallel. Be careful at chunk boundaries — an ORF or CpG island that spans a boundary could be missed or double-counted. Think about how to handle overlaps between chunks.

### Python `multiprocessing`
Use a `Pool` of workers within a single node to parallelize across cores. A caslake node has 48 cores — that's a lot of potential parallelism within a single job.

### SLURM Array Jobs
Assign each chunk or reading frame to an array task. Each runs independently, and you combine results with a dependency job.

### Multi-Node with `srun`
Request multiple nodes and use `srun` to distribute work, similar to Problem 5.

### Hybrid Approaches
Nothing stops you from combining techniques — for example, an array job where each task uses `multiprocessing` internally. Or a multi-node job where each node processes a different reading frame using all of its 48 cores.

### Algorithmic Improvements
Parallelism isn't the only way to go faster. Consider whether your serial code has inefficiencies that could be improved: redundant string operations, unnecessary copies, repeated work, or suboptimal data structures. Sometimes the biggest speedup comes from fixing the algorithm, not adding more hardware.


## Deliverables

Your `problem_7` directory should contain:

```
problem_7/
├── README.md
├── gene_finder_fast.py    (or whatever you name your optimized code)
├── *.sbatch               (all batch scripts used)
├── results/               (your output — candidate gene list)
└── benchmarks.md
```

### README.md

Your README should include:

- A description of your parallelization strategy and why you chose it
- Instructions on how to run your optimized gene finder from scratch (someone should be able to reproduce your results by following your README)
- Any dependencies or setup steps required

### benchmarks.md

Report your timing results in the following table:

| Approach           | Wall Time | Nodes | Tasks/Cores | Speedup vs. Serial |
| ------------------ | --------- | ----- | ----------- | ------------------- |
| Serial (baseline)  |           | 1     | 1           | 1.0x                |
| Your approach      |           |       |             |                     |

Include:

- The wall time of your **original serial implementation** as the baseline (run it as a single-core batch job on caslake to get a fair measurement)
- The wall time of your **optimized implementation**
- The speedup: $S = T_{serial} / T_{optimized}$
- If you tried multiple strategies, include a row for each and discuss which worked best and why

Answer the following questions:

- What was the primary bottleneck in the serial version?
- Which optimization had the biggest impact on performance?
- Did you hit diminishing returns? At what point did adding more parallelism stop helping?
- If you had access to 100 nodes, how would your approach change?

## Submitting Your Time

After your final run, record the job ID and wall time from `sacct`. Include this in your `benchmarks.md`:

```
Final Job ID: <your_job_id>
Final Wall Time: <HH:MM:SS>
```

Also, post the to `#showcase` on Slack.


**Submitting.** Put everything for this problem in `problem_7/` in your repository, with your results easy to find: your scripts, the output files your answers refer to, and `answers.md`. You can work in your clone on Midway3 and commit and push from there, or copy your work back to your laptop (for example with `scp`) and push from there. Either is fine.

Good luck — may the fastest gene finder win.
